#Import Library

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

#Load Clean Dataset

In [29]:
df_epo = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP/V2/Preprocessed_Dataset_Epo.csv')

df_llm = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP/V2/Preprocessed_Dataset_Llm.csv')

print(df_epo.shape)
print(df_llm.shape)

(6305, 3)
(24982, 3)


#Label Distribution

In [30]:
print(df_epo['emotion_label'].value_counts())

emotion_label
neutral    1988
happy      1274
angry      1129
sad        1003
anxious     911
Name: count, dtype: int64


#Dataset Ratio Configuration

In [31]:
TRAIN_RATIO = 0.70

VAL_RATIO = 0.15

TEST_RATIO = 0.15

SYNTHETIC_RATIO = 0.30

#Calculate Dataset Size

In [32]:
total_real = len(df_epo)

target_train_total = int(total_real * TRAIN_RATIO)

synthetic_count = int(target_train_total * SYNTHETIC_RATIO)

train_real_count = (target_train_total - synthetic_count)

print(f"Train Real : {train_real_count}")
print(f"Synthetic : {synthetic_count}")

Train Real : 3090
Synthetic : 1323


#Split Real Dataset

In [33]:
train_real, temp_real = train_test_split(df_epo, train_size=train_real_count, stratify=df_epo['emotion_label'], random_state=42)

#Split Validation & Test

In [34]:
val_real, test_real = train_test_split(temp_real, test_size=0.5, stratify=temp_real['emotion_label'], random_state=42)

#Minority Class Augmentation

In [35]:
class_counts = train_real['emotion_label'].value_counts()

max_class = class_counts.max()

synthetic_samples = []

for label, count in class_counts.items():
    deficit = max_class - count
    needed = min(deficit, synthetic_count)
    llm_subset = df_llm[df_llm['emotion_label'] == label].sample(n=needed, replace=True, random_state=42)
    synthetic_samples.append(llm_subset)

#Combine Final Train Dataset

In [36]:
df_llm_balanced = pd.concat(synthetic_samples, ignore_index=True)

train_final = pd.concat([train_real, df_llm_balanced], ignore_index=True)

train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

#Final Dataset Distribution

In [37]:
print(train_final.shape)

print(val_real.shape)

print(test_real.shape)

print(train_final['emotion_label'].value_counts())

(4870, 3)
(1607, 3)
(1608, 3)
emotion_label
anxious    974
happy      974
neutral    974
sad        974
angry      974
Name: count, dtype: int64


In [38]:
train_final.to_csv('train_dataset.csv', index=False)
val_real.to_csv('validation_dataset.csv', index=False)
test_real.to_csv('test_dataset.csv',index=False)

print("CSV files saved.")

CSV files saved.
